In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim 
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transform

transform = transform.Compose([
    transform.ToTensor(),
    transform.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [41:04<00:00, 69.2kB/s]
C:\Users\ASUS\anaconda3\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
train_loader = DataLoader(trainset , shuffle=True , batch_size=64)
test_loader = DataLoader(testset , batch_size=64)

### CNN Architechture

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),                      #(32,32,32)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2),    # Kernel_size=2 , stride_val=2          #(16,16,32)    
                                                                          #     |
            nn.Conv2d(32,64,kernel_size=3,padding=1),                     #(16,16,64)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2),                                            # (8,8,64) 
                                                                          #     |
            nn.Conv2d(64,128,kernel_size=3,padding=1),                    # (8,8,128)
            nn.ReLU(),                                                    #     |
            nn.MaxPool2d(2,2)    # Kernel_size=2 , stride_val=2           # (4,4,128)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(4*4*128 , 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layer(x)
        x = x.view(x.size(0),-1)  # Flattening 
        x = self.fc_layer(x)

        return x

In [17]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training model

In [20]:
epochs = 10
train_loss = []
test_loss = []

for epoch in range (epochs):
    running_train_loss = 0.0
    running_test_loss = 0.0
    model.train()

    for images ,labels in train_loader:
        optimizer.zero_grad()
        
        output = model.forward(images)
        loss = criterion(output , labels)
        loss.backward()
        optimizer.step()  
        running_train_loss += loss.item()
        
    epoch_train_loss = running_train_loss/len(train_loader)
    train_loss.append(epoch_train_loss)
        

    with torch.no_grad():
        model.eval()
        for images , labels in test_loader:
            
            output = model.forward(images)
            loss = criterion(output , labels)
            running_test_loss = loss.item()

        epoch_test_loss = running_test_loss/len(test_loader)
        test_loss.append(epoch_test_loss)

    print(f"epoch = {epoch+1} & Training loss:{epoch_train_loss} & Testing loss: {epoch_test_loss}")

epoch = 0 & Training loss:0.062771108971499 & Testing loss: 0.010957865198706366
epoch = 1 & Training loss:0.06393938505491528 & Testing loss: 0.014444080887326769
epoch = 2 & Training loss:0.052920262845110415 & Testing loss: 0.018122861339787768
epoch = 3 & Training loss:0.05463399334438205 & Testing loss: 0.01220513756867427
epoch = 4 & Training loss:0.06587045692609172 & Testing loss: 0.015386487268338538
epoch = 5 & Training loss:0.05697605912173179 & Testing loss: 0.012789076300942974
epoch = 6 & Training loss:0.05532067660425428 & Testing loss: 0.007646002587239454
epoch = 7 & Training loss:0.0466699677725609 & Testing loss: 0.0073648098927394596
epoch = 8 & Training loss:0.05870825400915565 & Testing loss: 0.00891315329606366
epoch = 9 & Training loss:0.04948546237430906 & Testing loss: 0.013502530990892155
